# **SureLink | Silver Transformation and Data Quality**

## 1. Import libraries and set up

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType)
from pyspark.sql.window import Window
from uuid import uuid4

EMAIL_PATTERN = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

print("SureLink Silver transformation started.")

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 3, Finished, Available, Finished, False)

SureLink Silver transformation started.


## 2. Load data from Bronze

In [2]:
bronze_customer_path = "abfss://SureLink@onelake.dfs.fabric.microsoft.com/SureLink_Bronze_lh.Lakehouse/Tables/dbo/Customer"
bronze_vehicle_path = "abfss://SureLink@onelake.dfs.fabric.microsoft.com/SureLink_Bronze_lh.Lakehouse/Tables/dbo/Vehicle"
bronze_policy_path = "abfss://SureLink@onelake.dfs.fabric.microsoft.com/SureLink_Bronze_lh.Lakehouse/Tables/dbo/Policy"
bronze_claim_path = "abfss://SureLink@onelake.dfs.fabric.microsoft.com/SureLink_Bronze_lh.Lakehouse/Tables/dbo/Claim"

customer_df = spark.read.format("delta").load(bronze_customer_path)
vehicle_df = spark.read.format("delta").load(bronze_vehicle_path)
policy_df = spark.read.format("delta").load(bronze_policy_path)
claim_df = spark.read.format("delta").load(bronze_claim_path)

bronze_tables = {
    "Customer": customer_df,
    "Vehicle": vehicle_df,
    "Policy": policy_df,
    "Claim": claim_df
}

for table_name, df in bronze_tables.items():
    print(f"{table_name}: {df.count():,} records")

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 4, Finished, Available, Finished, False)

Customer: 17,171 records
Vehicle: 20,597 records
Policy: 20,597 records
Claim: 19,931 records


In [3]:
print("Customer schema")
customer_df.printSchema()

print("Vehicle schema")
vehicle_df.printSchema()

print("Policy schema")
policy_df.printSchema()

print("Claim schema")
claim_df.printSchema()

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 5, Finished, Available, Finished, False)

Customer schema
root
 |-- customer_id: integer (nullable = true)
 |-- customer_ref: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- email: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- address_line: string (nullable = true)
 |-- city: string (nullable = true)
 |-- province: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- created_at: date (nullable = true)
 |-- last_modified: timestamp (nullable = true)

Vehicle schema
root
 |-- vehicle_id: integer (nullable = true)
 |-- make: string (nullable = true)
 |-- model: string (nullable = true)
 |-- model_year: short (nullable = true)
 |-- last_modified: timestamp (nullable = true)

Policy schema
root
 |-- policy_id: integer (nullable = true)
 |-- policy_number: string (nullable = true)
 |-- customer_id: in

## **Customer data**

### Check missing values

In [4]:
customer_missing = customer_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in customer_df.columns
])
display(customer_missing)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 488ca51d-f3ce-4772-a627-f7dc22dae739)

### Check duplicate customer IDs

In [5]:
duplicate_customer_ids = (
    customer_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)
display(duplicate_customer_ids)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09f73898-ae17-4d10-af71-15e2713ca606)

### Check empty text values

In [6]:
customer_text_columns = [
    "customer_ref", "first_name", "last_name", "gender", "email",
    "occupation", "education_level", "address_line", "city",
    "province", "postal_code"
]

customer_empty_values = customer_df.select([
    F.sum(F.when(
        F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1
    ).otherwise(0)).alias(c)
    for c in customer_text_columns
])
display(customer_empty_values)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4bb25124-e9ec-4315-bbf8-5981e61adae7)

### Check email formats

In [7]:
invalid_customer_emails = customer_df.filter(
    F.col("email").isNotNull()
    & ~F.col("email").rlike(EMAIL_PATTERN)
)
display(invalid_customer_emails.select(
    "customer_id", "first_name", "last_name", "email"
))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8d2edff5-817f-4d1e-81a0-0cb3c97d7547)

## Transform Customer data

In [8]:
## Trim text values, convert blank strings to NULL, standardise email addresses to lowercase
## Add flags to identify invalid emails and missing cities


customer_silver = customer_df

for column_name in customer_text_columns:
    customer_silver = customer_silver.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull() | (F.trim(F.col(column_name)) == ""),
            F.lit(None)
        ).otherwise(F.trim(F.col(column_name)))
    )

customer_silver = (
    customer_silver
    .withColumn("email", F.lower(F.col("email")))
    .withColumn("email_valid", F.col("email").rlike(EMAIL_PATTERN))
    .withColumn("city_missing", F.col("city").isNull())
)

display(customer_silver.select(
    "customer_id", "email", "email_valid", "city", "city_missing"
).limit(20))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9ab6c130-da62-4424-a6af-69e2b4915b4b)

### Check Customer quality flags

In [9]:
customer_silver.groupBy(
    "email_valid", "city_missing"
).count().orderBy(
    "email_valid", "city_missing"
).show()

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 11, Finished, Available, Finished, False)

+-----------+------------+-----+
|email_valid|city_missing|count|
+-----------+------------+-----+
|      false|       false|  170|
|       true|       false|16853|
|       true|        true|  148|
+-----------+------------+-----+



### Save and verify Customer

In [10]:
customer_silver.write.format("delta").mode("overwrite").saveAsTable("Customer")
silver_customer = spark.read.table("Customer")
print("Silver Customer rows:", silver_customer.count())
display(silver_customer.limit(10))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 12, Finished, Available, Finished, False)

Silver Customer rows: 17171


SynapseWidget(Synapse.DataFrame, 9e3898f8-1003-425f-8db5-b3ac34077190)

##  **Vehicle data**

### Check missing values

In [11]:
vehicle_missing = vehicle_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in vehicle_df.columns
])
display(vehicle_missing)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9a2371f4-8d65-4ec3-8473-f8914b6f4f00)

### Check duplicate vehicle IDs

In [12]:
duplicate_vehicle_ids = (
    vehicle_df.groupBy("vehicle_id")
    .count()
    .filter(F.col("count") > 1)
)
display(duplicate_vehicle_ids)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7a2b6bd-ae99-495f-a915-c37c4a46af13)

### Check empty make and model values

In [13]:
vehicle_text_columns = ["make", "model"]
vehicle_empty_values = vehicle_df.select([
    F.sum(F.when(
        F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1
    ).otherwise(0)).alias(c)
    for c in vehicle_text_columns
])
display(vehicle_empty_values)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 47826758-f6b4-44dc-8c80-484c731c676b)

## Transform Vehicle data

In [14]:
vehicle_silver = vehicle_df

for column_name in vehicle_text_columns:
    vehicle_silver = vehicle_silver.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull() | (F.trim(F.col(column_name)) == ""),
            F.lit(None)
        ).otherwise(F.trim(F.col(column_name)))
    )

vehicle_silver = (
    vehicle_silver
    .withColumn("make_missing", F.col("make").isNull())
    .withColumn("model_missing", F.col("model").isNull())
)

vehicle_silver.groupBy(
    "make_missing", "model_missing"
).count().show()

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 16, Finished, Available, Finished, False)

+------------+-------------+-----+
|make_missing|model_missing|count|
+------------+-------------+-----+
|       false|        false|20597|
+------------+-------------+-----+



### Save and verify Vehicle

In [15]:
vehicle_silver.write.format("delta").mode("overwrite").saveAsTable("Vehicle")
silver_vehicle = spark.read.table("Vehicle")
print("Silver Vehicle rows:", silver_vehicle.count())
display(silver_vehicle.limit(10))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 17, Finished, Available, Finished, False)

Silver Vehicle rows: 20597


SynapseWidget(Synapse.DataFrame, 654752c2-ebe9-4c6f-bb09-7087ae7feeec)

## **Policy data**

### Check missing values

In [16]:
policy_missing = policy_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in policy_df.columns
])
display(policy_missing)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8a2ee7c9-a8dc-4adc-83e6-28d032919020)

### Check duplicate policy IDs

In [17]:
duplicate_policy_ids = (
    policy_df.groupBy("policy_id")
    .count()
    .filter(F.col("count") > 1)
)
display(duplicate_policy_ids)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df200975-ec75-466b-b6ea-61e4fe7a9219)

### Check empty text values

In [18]:
policy_text_columns = ["policy_number", "cover_level", "policy_status"]
policy_empty_values = policy_df.select([
    F.sum(F.when(
        F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1
    ).otherwise(0)).alias(c)
    for c in policy_text_columns
])
display(policy_empty_values)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 067571d2-db8b-4d3a-a773-4a2dc758e56f)

### Investigate negative liability limits

Found 21 negative `extended_liability_limit` values, all -1,000,000 and associated with THIRD PARTY policies. These values are kept unchanged until the business confirms the intended rule. In the validation section they are recorded as a high-severity issue.

In [19]:
negative_liability_limits = policy_df.filter(
    F.col("extended_liability_limit") < 0
)
print("Negative liability limit records:", negative_liability_limits.count())
display(negative_liability_limits.select(
    "policy_id", "policy_number", "cover_level",
    "policy_status", "extended_liability_limit"
))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 21, Finished, Available, Finished, False)

Negative liability limit records: 21


SynapseWidget(Synapse.DataFrame, 50a9f055-560a-41b9-9e83-741a46edcdef)

## Transform Policy data

In [20]:
policy_silver = policy_df

for column_name in policy_text_columns:
    policy_silver = policy_silver.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull() | (F.trim(F.col(column_name)) == ""),
            F.lit(None)
        ).otherwise(F.trim(F.col(column_name)))
    )

policy_silver = (
    policy_silver
    .withColumn("cover_level", F.upper(F.col("cover_level")))
    .withColumn("policy_status", F.upper(F.col("policy_status")))
)

display(policy_silver.select(
    "policy_number", "cover_level", "policy_status",
    "extended_liability_limit"
).limit(10))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7aa19198-edcf-4246-8d62-6d4b0b2f86f6)

### Save and verify Policy

In [21]:
policy_silver.write.format("delta").mode("overwrite").saveAsTable("Policy")
silver_policy = spark.read.table("Policy")
print("Silver Policy rows:", silver_policy.count())
display(silver_policy.limit(10))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 23, Finished, Available, Finished, False)

Silver Policy rows: 20597


SynapseWidget(Synapse.DataFrame, 60724756-0a93-4f76-90b2-0a0cac7b8dd0)

## **Claim data**

### Check missing values

In [22]:
claim_missing = claim_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in claim_df.columns
])
display(claim_missing)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0fc7c2ef-353b-44f9-822d-7a30c3e0e4ca)

### Check duplicate claim IDs

In [23]:
duplicate_claim_ids = (
    claim_df.groupBy("claim_id")
    .count()
    .filter(F.col("count") > 1)
)
display(duplicate_claim_ids)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5079e76f-78aa-4669-a640-6ad009bbfade)

### Check missing approved amounts by claim status

Analysis show missing approved amounts for REJECTED, SUBMITTED and UNDER REVIEW claims. Their meaning depends on the insurance business rules and claim lifecycle.

In [24]:
## Preserve NULL values

missing_approval_by_status = (
    claim_df.filter(F.col("approved_amount").isNull())
    .groupBy("claim_status")
    .count()
    .orderBy("claim_status")
)
display(missing_approval_by_status)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c1bfc2ea-bce9-48a9-9c3e-0c96cdd051b1)

### Check empty text values

In [25]:
claim_text_columns = [
    "claim_number", "incident_type", "collision_type", "incident_severity",
    "authorities_contacted", "incident_province", "incident_city",
    "incident_location", "property_damage", "police_report_available",
    "claim_status", "fraud_reported", "claim_notes"
]
claim_empty_values = claim_df.select([
    F.sum(F.when(
        F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1
    ).otherwise(0)).alias(c)
    for c in claim_text_columns
])
display(claim_empty_values)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 70890672-ccdf-428d-bad2-6c94658e81ab)

### Check missing authorities-contacted values

Analysis found 1,792 missing values, concentrated in PARKED CAR and VEHICLE THEFT claims.

In [26]:
##Keep them NULL pending confirmation of whether this field is mandatory for every claim type

missing_authorities_by_incident = (
    claim_df.filter(F.col("authorities_contacted").isNull())
    .groupBy("incident_type")
    .count()
    .orderBy(F.desc("count"))
)
display(missing_authorities_by_incident)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 754b5a89-4897-42da-a57e-a6334554452b)

## Transform Claim data

In [27]:
claim_silver = claim_df

for column_name in claim_text_columns:
    claim_silver = claim_silver.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull() | (F.trim(F.col(column_name)) == ""),
            F.lit(None)
        ).otherwise(F.trim(F.col(column_name)))
    )

claim_category_columns = [
    "incident_type", "collision_type", "incident_severity",
    "authorities_contacted", "claim_status", "fraud_reported"
]

for column_name in claim_category_columns:
    claim_silver = claim_silver.withColumn(
        column_name, F.upper(F.col(column_name))
    )

display(claim_silver.select(
    "claim_number", "incident_type", "incident_severity",
    "authorities_contacted", "claim_status", "approved_amount"
).limit(10))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c005b6d-2e45-44ea-83ce-cef487b56274)

### Save and verify Claim

In [28]:
claim_silver.write.format("delta").mode("overwrite").saveAsTable("Claim")
silver_claim = spark.read.table("Claim")
print("Silver Claim rows:", silver_claim.count())
display(silver_claim.limit(10))

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 30, Finished, Available, Finished, False)

Silver Claim rows: 19931


SynapseWidget(Synapse.DataFrame, 690b9541-b9d8-4c25-8809-5940151c8d6d)

## Verify all Silver tables

In [29]:
## Compare the saved Silver counts with the Bronze counts

## The expected row counts:
## Customer: 17,17, 
## Vehicle:  20,597 
## Policy:   20,597 
## Claim:    19,931

silver_tables = {
    "Customer": spark.read.table("Customer"),
    "Vehicle": spark.read.table("Vehicle"),
    "Policy": spark.read.table("Policy"),
    "Claim": spark.read.table("Claim")
}

for table_name, silver_df in silver_tables.items():
    bronze_count = bronze_tables[table_name].count()
    silver_count = silver_df.count()
    print(
        f"{table_name}: Bronze={bronze_count:,}, "
        f"Silver={silver_count:,}, "
        f"Match={bronze_count == silver_count}"
    )

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 31, Finished, Available, Finished, False)

Customer: Bronze=17,171, Silver=17,171, Match=True
Vehicle: Bronze=20,597, Silver=20,597, Match=True
Policy: Bronze=20,597, Silver=20,597, Match=True
Claim: Bronze=19,931, Silver=19,931, Match=True


## Data quality function

The first function evaluates one rule against one DataFrame. It returns the table name, column, rule, total records, failed records, severity, status and description. A PASS means no records failed the condition. A FAIL means at least one record failed.

In [30]:
def run_quality_check(
    df,
    table_name,
    column_name,
    rule_name,
    condition,
    severity,
    description
):
    total_records = df.count()
    failed_records = df.filter(condition).count()
    status = "FAIL" if failed_records > 0 else "PASS"

    return (
        table_name,
        column_name,
        rule_name,
        total_records,
        failed_records,
        severity,
        status,
        description
    )

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 32, Finished, Available, Finished, False)

### Save validation history function

Each validation execution receives one run ID and timestamp. Results are appended to `DataQualityResults`, preserving earlier runs. Schema merging supports the `total_records` column added to the table after the first version.

In [31]:
def save_quality_results(check_results):
    run_id = str(uuid4())

    result_schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("column_name", StringType(), False),
        StructField("rule_name", StringType(), False),
        StructField("total_records", LongType(), False),
        StructField("failed_records", LongType(), False),
        StructField("severity", StringType(), False),
        StructField("status", StringType(), False),
        StructField("description", StringType(), False)
    ])

    rows = [(run_id,) + result for result in check_results]
    results_df = spark.createDataFrame(rows, result_schema)
    results_df = results_df.withColumn(
        "run_timestamp", F.current_timestamp()
    )

    (results_df.write
        .format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .saveAsTable("DataQualityResults"))

    print("Validation results saved.")
    print("Run ID:", run_id)
    return results_df

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 33, Finished, Available, Finished, False)

## Data quality checks

Run all six rules together so the results share the same run ID and timestamp. The conditions below use the quality flags created during the Customer and Vehicle transformations. Policy and Claim rules check the preserved source values.

In [32]:
customer = spark.read.table("Customer")
vehicle = spark.read.table("Vehicle")
policy = spark.read.table("Policy")
claim = spark.read.table("Claim")

check_results = [
    run_quality_check(
        customer, "Customer", "email", "VALID_EMAIL_FORMAT",
        F.col("email_valid") == F.lit(False), "MEDIUM",
        "Email address does not match the expected format."
    ),
    run_quality_check(
        customer, "Customer", "city", "CITY_REQUIRED",
        F.col("city_missing") == F.lit(True), "LOW",
        "Customer city is missing."
    ),
    run_quality_check(
        vehicle, "Vehicle", "make", "VEHICLE_MAKE_REQUIRED",
        F.col("make_missing") == F.lit(True), "MEDIUM",
        "Vehicle make is missing."
    ),
    run_quality_check(
        vehicle, "Vehicle", "model", "VEHICLE_MODEL_REQUIRED",
        F.col("model_missing") == F.lit(True), "MEDIUM",
        "Vehicle model is missing."
    ),
    run_quality_check(
        policy, "Policy", "extended_liability_limit",
        "NON_NEGATIVE_LIABILITY_LIMIT",
        F.col("extended_liability_limit") < 0, "HIGH",
        "Liability limit is negative and requires business investigation."
    ),
    run_quality_check(
        claim, "Claim", "authorities_contacted",
        "AUTHORITIES_CONTACTED_REQUIRED",
        F.col("authorities_contacted").isNull(), "MEDIUM",
        "Authorities contacted is missing. Confirm whether this field is mandatory for every claim type."
    )
]

for result in check_results:
    print(result)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 34, Finished, Available, Finished, False)

('Customer', 'email', 'VALID_EMAIL_FORMAT', 17171, 170, 'MEDIUM', 'FAIL', 'Email address does not match the expected format.')
('Customer', 'city', 'CITY_REQUIRED', 17171, 148, 'LOW', 'FAIL', 'Customer city is missing.')
('Vehicle', 'make', 'VEHICLE_MAKE_REQUIRED', 20597, 0, 'MEDIUM', 'PASS', 'Vehicle make is missing.')
('Vehicle', 'model', 'VEHICLE_MODEL_REQUIRED', 20597, 0, 'MEDIUM', 'PASS', 'Vehicle model is missing.')
('Policy', 'extended_liability_limit', 'NON_NEGATIVE_LIABILITY_LIMIT', 20597, 21, 'HIGH', 'FAIL', 'Liability limit is negative and requires business investigation.')
('Claim', 'authorities_contacted', 'AUTHORITIES_CONTACTED_REQUIRED', 19931, 1792, 'MEDIUM', 'FAIL', 'Authorities contacted is missing. Confirm whether this field is mandatory for every claim type.')


### Save validation run

In [33]:
saved_results = save_quality_results(check_results)
saved_results.show(truncate=False)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 35, Finished, Available, Finished, False)

Validation results saved.
Run ID: 1927ed62-1ea7-4b23-b504-a0f330d74a91
+------------------------------------+----------+------------------------+------------------------------+-------------+--------------+--------+------+-----------------------------------------------------------------------------------------------+--------------------------+
|run_id                              |table_name|column_name             |rule_name                     |total_records|failed_records|severity|status|description                                                                                    |run_timestamp             |
+------------------------------------+----------+------------------------+------------------------------+-------------+--------------+--------+------+-----------------------------------------------------------------------------------------------+--------------------------+
|1927ed62-1ea7-4b23-b504-a0f330d74a91|Customer  |email                   |VALID_EMAIL_FORMAT            |17

## Retrieve latest result for each rule

The results table stores multiple executions. This query returns only the most recent result for each table and rule. The run ID is used as a secondary sort value if timestamps match.

In [34]:
data_quality = spark.read.table("DataQualityResults")

latest_window = Window.partitionBy(
    "table_name", "rule_name"
).orderBy(
    F.desc("run_timestamp"), F.desc("run_id")
)

latest_quality_results = (
    data_quality
    .withColumn("row_num", F.row_number().over(latest_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

latest_quality_results.select(
    "table_name", "rule_name", "total_records", "failed_records",
    "severity", "status", "run_timestamp"
).orderBy("table_name", "rule_name").show(truncate=False)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 36, Finished, Available, Finished, False)

+----------+------------------------------+-------------+--------------+--------+------+--------------------------+
|table_name|rule_name                     |total_records|failed_records|severity|status|run_timestamp             |
+----------+------------------------------+-------------+--------------+--------+------+--------------------------+
|Claim     |AUTHORITIES_CONTACTED_REQUIRED|19931        |1792          |MEDIUM  |FAIL  |2026-11-15 15:19:51.800043|
|Customer  |CITY_REQUIRED                 |17171        |148           |LOW     |FAIL  |2026-11-15 15:19:51.800043|
|Customer  |VALID_EMAIL_FORMAT            |17171        |170           |MEDIUM  |FAIL  |2026-11-15 15:19:51.800043|
|Policy    |NON_NEGATIVE_LIABILITY_LIMIT  |20597        |21            |HIGH    |FAIL  |2026-11-15 15:19:51.800043|
|Vehicle   |VEHICLE_MAKE_REQUIRED         |20597        |0             |MEDIUM  |PASS  |2026-11-15 15:19:51.800043|
|Vehicle   |VEHICLE_MODEL_REQUIRED        |20597        |0             |

## 15. Summarise data quality

The total rule failures are the sum of failures across all rules. They are not necessarily unique records because one record might fail more than one rule.

In [35]:
quality_summary = latest_quality_results.agg(
    F.count("*").alias("total_checks"),
    F.sum(F.when(F.col("status") == "PASS", 1).otherwise(0)).alias("passed_checks"),
    F.sum(F.when(F.col("status") == "FAIL", 1).otherwise(0)).alias("failed_checks"),
    F.sum("failed_records").alias("total_rule_failures")
)
quality_summary.show(truncate=False)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 37, Finished, Available, Finished, False)

+------------+-------------+-------------+-------------------+
|total_checks|passed_checks|failed_checks|total_rule_failures|
+------------+-------------+-------------+-------------------+
|6           |2            |4            |2131               |
+------------+-------------+-------------+-------------------+



### Break down results by severity

In [36]:
severity_summary = (
    latest_quality_results
    .groupBy("severity", "status")
    .agg(
        F.count("*").alias("number_of_checks"),
        F.sum("failed_records").alias("failed_records")
    )
    .orderBy("severity", "status")
)
severity_summary.show(truncate=False)

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 38, Finished, Available, Finished, False)

+--------+------+----------------+--------------+
|severity|status|number_of_checks|failed_records|
+--------+------+----------------+--------------+
|HIGH    |FAIL  |1               |21            |
|LOW     |FAIL  |1               |148           |
|MEDIUM  |FAIL  |2               |1962          |
|MEDIUM  |PASS  |2               |0             |
+--------+------+----------------+--------------+



## Final checks

In [37]:
print("Silver table counts")
for table_name in ["Customer", "Vehicle", "Policy", "Claim"]:
    print(f"{table_name}: {spark.read.table(table_name).count():,}")

print("Latest validation checks:", latest_quality_results.count())
print("Validation history rows:", spark.read.table("DataQualityResults").count())
print("SureLink Silver transformation and data quality run completed.")

StatementMeta(, b33760d1-73f0-43ae-b452-bee3c794bdb0, 39, Finished, Available, Finished, False)

Silver table counts
Customer: 17,171
Vehicle: 20,597
Policy: 20,597
Claim: 19,931
Latest validation checks: 6
Validation history rows: 14
SureLink Silver transformation and data quality run completed.


In [2]:
# Read the registered data quality table
dq_df = spark.table("dbo.dataqualityresults")

# Display its structure
dq_df.printSchema()

# Count the records
print("Total quality results:", dq_df.count())

# Display the records
display(dq_df.orderBy("run_timestamp", ascending=False))

StatementMeta(, 9a8738ff-6335-4553-b66d-9c9096c10701, 4, Submitted, Running, Running, True)